In [ ]:
!pip install catboost


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.4 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd


from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [ ]:
TRAIN = "/content/train.csv"
TEST = "/content/test.csv"

train_df = pd.read_csv(TRAIN)
test_df = pd.read_csv(TEST)

In [ ]:
X = train_df.drop(columns=["Age"])
y = train_df["Age"]

In [ ]:
numeric_features = X.select_dtypes(include = ['int64','float64']).columns
categorical_features = X.select_dtypes(include = ['object']).columns

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features),
        ("num", "passthrough", numeric_features)
      ]
  )

In [ ]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42 )

In [ ]:
reg_rf = RandomForestRegressor(n_estimators=200, random_state=42)
reg_gb = GradientBoostingRegressor(n_estimators=200, random_state=42)
reg_cb = CatBoostRegressor(n_estimators=200, random_state=42, verbose=False)

In [ ]:
model_to_train = {
    'Random Forest' : reg_rf,
    'Gradient Boosting': reg_gb,
    'Cat Boosting': reg_cb
}

In [ ]:
result = []

for name, model in model_to_train.items():
  if name == 'Cat Boosting':
    model.fit(X_train, y_train, cat_features=['Sex'])
    y_pred = model.predict(X_test)
  else:
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

  r2 = r2_score(y_test, y_pred)
  rmse = np.sqrt(mean_squared_error(y_test, y_pred))
  mae = mean_absolute_error(y_test, y_pred)

  result.append({
      "Model": name,
      "R2 Score": r2,
      "RMSE": rmse,
      "MAE": mae
  })

results_df = pd.DataFrame(result).sort_values("R2 Score", ascending=False)
print(results_df)

               Model  R2 Score      RMSE       MAE
1  Gradient Boosting  0.610079  2.032604  1.379895
2       Cat Boosting  0.605663  2.044082  1.396146
0      Random Forest  0.598528  2.062490  1.422437


In [ ]:
param_grid_gb = {
    'model__n_estimators': [200, 400, 600],
    'model__learning_rate': [0.03, 0.05, 0.1],
    'model__max_depth': [2, 3, 4],
    'model__subsample': [0.8, 1.0],
    'model__min_samples_split': [2, 5, 10]
}


gb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(random_state=42))
])


grid = GridSearchCV(
    estimator=gb_pipeline,
    param_grid=param_grid_gb,
    scoring='r2',
    cv=5,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best Score:", grid.best_score_)

Fitting 5 folds for each of 162 candidates, totalling 810 fits
Best Parameters: {'model__learning_rate': 0.03, 'model__max_depth': 3, 'model__min_samples_split': 10, 'model__n_estimators': 600, 'model__subsample': 0.8}
Best Score: 0.6137200467677094


In [ ]:
X_final_test = test_df.copy()

best_model = grid.best_estimator_

test_pred = best_model.predict(X_final_test)